# Somali News Dataset — Maximum Cleaning Notebook

This notebook combines:

1. The **advanced site-specific cleaning notebook** we created recently.
2. The **previous cleaning notebook logic**, especially the stronger non-Somali / English article removal step.

Main outputs:

- `data/processed/cleaned_dataset_maximum.csv`
- `data/processed/suspicious_rows_for_review.csv`
- `data/processed/removed_non_somali_articles.csv`
- `data/processed/removed_rows_all_reasons.csv`

Run the cells from top to bottom.


In [1]:

# CELL 1 — OPTIONAL DEPENDENCY CHECK
# ==================================
# The notebook can run with only pandas.
# For stronger English/non-Somali detection, install langdetect and langid.
# For Excel export, install openpyxl.
#
# Recommended Anaconda Prompt commands:
#   conda activate somali_cleaner
#   conda install pandas openpyxl -y
#   pip install langdetect langid
#
# Or run this notebook cell to install the optional packages:

import sys
import importlib.util
import subprocess

OPTIONAL_PACKAGES = ["langdetect", "langid", "openpyxl"]

missing = [
    pkg for pkg in OPTIONAL_PACKAGES
    if importlib.util.find_spec(pkg) is None
]

if missing:
    print("⚠️ Missing optional packages:", missing)
    print("Installing missing packages now...")
    for pkg in missing:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
            print(f"✅ Installed: {pkg}")
        except Exception as e:
            print(f"❌ Could not install {pkg}: {e}")
            print(f"   You can install manually later with: pip install {pkg}")
else:
    print("✅ All optional packages are already installed.")


✅ All optional packages are already installed.


In [2]:

# CELL 2 — IMPORTS, CONFIGURATION, AND LOAD COMBINED RAW DATA
# ==========================================================

import pandas as pd
import numpy as np
import re
import html
import hashlib
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import importlib.util

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REVIEW_DIR = DATA_DIR / "review"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Possible locations of your combined raw dataset
COMBINED_RAW_CANDIDATES = [
    Path("data/combined_raw_data.csv"),
    Path("combined_raw_data.csv"),
    Path("data/combined/combined_raw_data.csv"),
    Path("data/raw/combined_raw_data.csv"),
]

combined_path = None

for p in COMBINED_RAW_CANDIDATES:
    if p.exists():
        combined_path = p
        break

if combined_path is None:
    raise FileNotFoundError(
        "❌ Could not find combined_raw_data.csv.\n"
        "Put it in one of these locations:\n"
        "  - data/combined_raw_data.csv\n"
        "  - combined_raw_data.csv\n"
        "  - data/combined/combined_raw_data.csv"
    )

raw_df = pd.read_csv(combined_path, dtype=str, keep_default_na=False)

print(f"✅ Loaded combined raw dataset from: {combined_path.resolve()}")
print(f"📊 Raw rows: {len(raw_df):,}")
print(f"📊 Raw columns: {len(raw_df.columns):,}")

# ── Standardize expected column names ────────────────────────────────────────
raw_df.columns = (
    raw_df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

rename_map = {
    "title": "headline",
    "content": "body",
    "article_title": "headline",
    "article_body": "body",
    "link": "url",
    "source_site": "site",
    "file_category": "category_from_file",
}

for old_col, new_col in rename_map.items():
    if old_col in raw_df.columns and new_col not in raw_df.columns:
        raw_df = raw_df.rename(columns={old_col: new_col})

# Required columns
for col in ["site", "source", "category", "url", "headline", "body", "source_file"]:
    if col not in raw_df.columns:
        raw_df[col] = ""

# If site is missing, use source
raw_df["site"] = raw_df["site"].astype(str).str.strip().str.lower()
raw_df["source"] = raw_df["source"].astype(str).str.strip().str.lower()

raw_df.loc[raw_df["site"].eq("") & raw_df["source"].ne(""), "site"] = raw_df.loc[
    raw_df["site"].eq("") & raw_df["source"].ne(""),
    "source"
]

# Infer site/category from source_file if still missing
def infer_site_category_from_filename(filename: str):
    stem = Path(str(filename)).stem.strip().lower()

    if "__" in stem:
        site, category = stem.split("__", 1)
    else:
        parts = stem.split("_")
        category = parts[-1] if len(parts) > 1 else ""
        site = "_".join(parts[:-1]) if len(parts) > 1 else stem

    return site.strip(), category.strip()


for idx, row in raw_df.iterrows():
    if (not str(row.get("site", "")).strip() or not str(row.get("category", "")).strip()) and str(row.get("source_file", "")).strip():
        inferred_site, inferred_category = infer_site_category_from_filename(row["source_file"])

        if not str(row.get("site", "")).strip():
            raw_df.at[idx, "site"] = inferred_site

        if not str(row.get("category", "")).strip():
            raw_df.at[idx, "category"] = inferred_category

# Final basic strip
for col in ["site", "source", "category", "url", "headline", "body", "source_file"]:
    raw_df[col] = raw_df[col].astype(str).str.strip()

print("\n📌 Sources found:")
display(raw_df["site"].value_counts().rename_axis("site").reset_index(name="rows"))

print("\n📌 Categories found:")
display(raw_df["category"].value_counts().rename_axis("category").reset_index(name="rows"))

display(raw_df.head())


✅ Loaded combined raw dataset from: E:\somali_cleaner\data\combined_raw_data.csv
📊 Raw rows: 35,797
📊 Raw columns: 9

📌 Sources found:


,site,rows
0,goobjoog,12860
1,caasimada,9365
2,kooxda,7193
3,mustaqbalmedia,4640
4,laacibnet,1094
5,bbc_somali,245
6,puntlandpost,226
7,kubadlive,114
8,sonna,26
9,wararka24,15



📌 Categories found:


,category,rows
0,caalamka,10116
1,siyaasad,10062
2,ciyaaro,9024
3,amni,6595


,id,site,category,url,headline,body,scraped_at,word_count,source_file,source
0,bbc_somali__ciyaaro__000001,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/clyp28rnzw0o,Maxay dalalku ka faa'idaan marka ay u soo baxa...,"Xigashada Sawirka, Getty Images Dalalka si guu...",2026-05-19 20:30:29,656,bbc_somali__ciyaaro.csv,
1,bbc_somali__ciyaaro__000002,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cd9plkkj1pdo,Maxaa ka jira in ciyaartooyda kooxdii caanka a...,"Xigashada Sawirka, Getty Images Kooxda Real Ma...",2026-05-19 20:30:33,809,bbc_somali__ciyaaro.csv,
2,bbc_somali__ciyaaro__000003,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cjdplv01z52o,Ciyaaryahanada ugu sarreeya Afrika ee aan ka q...,"Xigashada Sawirka, Getty Images Iyada oo ay so...",2026-05-19 20:30:36,616,bbc_somali__ciyaaro.csv,
3,bbc_somali__ciyaaro__000004,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/c332veek778o,Maxaa ugu wacan Arsenal inaysan weligeed ku gu...,"Xigashada Sawirka, Getty Images Arsenal walige...",2026-05-19 20:30:39,848,bbc_somali__ciyaaro.csv,
4,bbc_somali__ciyaaro__000005,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cly07pjzdm0o,Dhamaan xogta iyo macluumaadka aad u baahan ta...,"Xigashada Sawirka, Getty Images Iyadoo dhamaan...",2026-05-19 20:30:42,1731,bbc_somali__ciyaaro.csv,


In [3]:

# CELL 3 — RAW DATA QUALITY REPORT
# ================================

df_report = raw_df.copy()

df_report["body_words"] = df_report["body"].astype(str).str.split().str.len()
df_report["body_chars"] = df_report["body"].astype(str).str.len()
df_report["headline_chars"] = df_report["headline"].astype(str).str.len()
df_report["link_count"] = df_report["body"].astype(str).str.count(r"https?://|www\.")

print("═" * 90)
print("RAW DATASET OVERVIEW")
print("═" * 90)
print(f"Total rows: {len(df_report):,}")
print(f"Total sources: {df_report['site'].nunique():,}")
print(f"Total categories: {df_report['category'].nunique():,}")

print("\n📌 Missing / empty values:")
missing_report = []

for col in ["url", "headline", "body", "category", "site"]:
    missing_report.append({
        "column": col,
        "empty_count": int((df_report[col].astype(str).str.strip() == "").sum()),
        "empty_percent": round((df_report[col].astype(str).str.strip() == "").mean() * 100, 2),
    })

display(pd.DataFrame(missing_report))

print("\n📌 Body length statistics by source:")
display(
    df_report
    .groupby("site")["body_words"]
    .describe()
    .round(1)
    .reset_index()
)

print("\n📌 Duplicate report:")
duplicate_report = {
    "duplicate_url_rows": int(df_report[df_report["url"].duplicated(keep=False)].shape[0]),
    "unique_duplicate_urls": int(df_report[df_report["url"].duplicated(keep=False)]["url"].nunique()),
    "duplicate_exact_body_rows": int(df_report[df_report["body"].duplicated(keep=False)].shape[0]),
    "very_short_bodies_under_50_words": int((df_report["body_words"] < 50).sum()),
    "very_long_bodies_over_1000_words": int((df_report["body_words"] > 1000).sum()),
    "rows_with_links_in_body": int((df_report["link_count"] > 0).sum()),
}

display(pd.DataFrame([duplicate_report]).T.rename(columns={0: "count"}))

print("\n📌 Important noise pattern counts by source:")

PATTERN_CHECKS = {
    "bbc_xigashada_sawirka": r"Xigashada Sawirka",
    "bbc_end_of_content": r"End of content",
    "bbc_ugu_akhris_badan": r"Ugu akhris badan",
    "bbc_whatsapp_ad": r"WhatsApp|Halkaan kaga soo biir|Dhamaadka xayeysiinta",

    "caasimada_share_dateline": r"^Share\s+",
    "caasimada_read_more": r"Read more",
    "caasimada_online": r"Caasimada\s+Online",
    "caasimada_isha": r"Isha\s*:",
    "caasimada_wq_credit": r"W/[QT]\s*:",
    "caasimada_email": r"\[email|email\s+protected",

    "goobjoog_news": r"Goobjoog\s+News",
    "goobjoog_halkaan_hoose": r"Halkaan\s+hoose|Hoos\s+Ka\s+Dhageyso",
    "goobjoog_wariye": r"Wariye\s*:",

    "mustaqbal_tail_or_name": r"\bMustaqbal\b",
    "mustaqbal_isha": r"Isha\s*:",
    "mustaqbal_lasoco": r"La\s+soco\s+wixii",

    "kooxda_site": r"Kooxda\.com",
    "kubadlive_views": r"Views?\s*:\s*\d+",
    "laacibnet_comment_form": r"Save my name, email, and website in this browser",
    "puntlandpost_tail": r"PUNTLAND\s+POST",
    "sonna_by_credit": r"\bBy\.?\s+",
    "wararka24_source": r"Wararka24",
    "garowe_boilerplate": r"Advertise with GaroweOnline|manage your cookie choices|Copyright © GAROWONLINE",

    "policy_contact_pages": r"Privacy Policy|Terms and Conditions|Editorial Policy|Cookie Policy|Contact Us|nala soo xiriir|Disclaimer",
    "contact_form": r"Your name\s+Your email\s+Subject",
}

pattern_rows = []

for site, sub in df_report.groupby("site"):
    for pattern_name, pattern in PATTERN_CHECKS.items():
        body_count = sub["body"].astype(str).str.contains(pattern, case=False, regex=True).sum()
        headline_count = sub["headline"].astype(str).str.contains(pattern, case=False, regex=True).sum()
        total = int(body_count + headline_count)

        if total > 0:
            pattern_rows.append({
                "site": site,
                "pattern": pattern_name,
                "count": total,
                "percent_of_site": round(total / len(sub) * 100, 2),
            })

pattern_report = pd.DataFrame(pattern_rows)

if not pattern_report.empty:
    pattern_report = pattern_report.sort_values(["site", "count"], ascending=[True, False])
    display(pattern_report)
else:
    print("No configured noise patterns found.")


══════════════════════════════════════════════════════════════════════════════════════════
RAW DATASET OVERVIEW
══════════════════════════════════════════════════════════════════════════════════════════
Total rows: 35,797
Total sources: 12
Total categories: 4

📌 Missing / empty values:


,column,empty_count,empty_percent
0,url,0,0.0
1,headline,0,0.0
2,body,0,0.0
3,category,0,0.0
4,site,0,0.0



📌 Body length statistics by source:


,site,count,mean,std,min,25%,50%,75%,max
0,bbc_somali,245.0,662.6,302.1,20.0,472.0,645.0,831.0,1932.0
1,caasimada,9365.0,386.5,231.9,13.0,248.0,310.0,436.0,2692.0
2,garoweonline,11.0,232.0,61.7,46.0,250.0,250.0,250.0,256.0
3,goobjoog,12860.0,177.4,153.0,9.0,102.0,138.0,198.0,2392.0
4,kooxda,7193.0,170.8,119.4,12.0,114.0,145.0,190.0,2453.0
5,kooxdamanta,8.0,329.1,85.7,140.0,321.5,340.5,373.2,429.0
6,kubadlive,114.0,287.8,159.1,10.0,171.0,259.5,371.5,941.0
7,laacibnet,1094.0,218.7,187.1,91.0,160.0,185.0,220.0,2577.0
8,mustaqbalmedia,4640.0,218.8,111.9,8.0,162.0,193.0,239.0,1655.0
9,puntlandpost,226.0,207.3,113.6,44.0,139.0,189.0,240.8,1072.0



📌 Duplicate report:


,count
duplicate_url_rows,1407
unique_duplicate_urls,699
duplicate_exact_body_rows,1442
very_short_bodies_under_50_words,411
very_long_bodies_over_1000_words,432
rows_with_links_in_body,119



📌 Important noise pattern counts by source:


,site,pattern,count,percent_of_site
2,bbc_somali,bbc_ugu_akhris_badan,225,91.84
0,bbc_somali,bbc_xigashada_sawirka,222,90.61
3,bbc_somali,bbc_whatsapp_ad,155,63.27
1,bbc_somali,bbc_end_of_content,148,60.41
5,bbc_somali,mustaqbal_tail_or_name,6,2.45
...,...,...,...,...
59,sonna,mustaqbal_tail_or_name,2,7.69
63,wararka24,wararka24_source,10,66.67
64,wararka24,policy_contact_pages,4,26.67
62,wararka24,sonna_by_credit,2,13.33


In [4]:

# CELL 4 — MAXIMUM CLEANING FUNCTIONS
# ===================================
# This cell combines:
#   - Advanced website-specific cleaning functions
#   - Previous notebook's strong non-Somali / English article removal logic
#   - Safe validation helpers

# ─────────────────────────────────────────────────────────────────────────────
# Optional language detection libraries
# ─────────────────────────────────────────────────────────────────────────────

try:
    from langdetect import detect, LangDetectException
except Exception:
    detect = None

    class LangDetectException(Exception):
        pass

try:
    import langid
except Exception:
    langid = None


# ─────────────────────────────────────────────────────────────────────────────
# Cleaning thresholds
# ─────────────────────────────────────────────────────────────────────────────

MIN_HEADLINE_CHARS = 8
MAX_HEADLINE_CHARS = 180
MIN_BODY_WORDS_FINAL = 50
MAX_BODY_WORDS_REVIEW = 1200

VALID_CATEGORIES = {
    "siyaasad",
    "ciyaaro",
    "caafimaad",
    "caalamka",
    "amni",
}


# ─────────────────────────────────────────────────────────────────────────────
# General text helpers
# ─────────────────────────────────────────────────────────────────────────────

def safe_text(x) -> str:
    """Converts missing or non-string values into safe strings."""
    if pd.isna(x):
        return ""
    return str(x)


def normalize_text(text: str) -> str:
    """
    Safe general text normalization.
    Keeps useful Somali text while removing invisible characters and spacing issues.
    """
    text = safe_text(text)
    text = html.unescape(text)

    # Remove invisible characters
    text = re.sub(r"[\u200b\u200c\u200d\ufeff\u00ad]", "", text)

    # Normalize whitespace
    text = re.sub(r"[\r\n\t\xa0]+", " ", text)

    # Fix punctuation spacing
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # Collapse repeated punctuation conservatively
    text = re.sub(r"([!?]){3,}", r"\1\1", text)
    text = re.sub(r"([.]){4,}", "...", text)

    # Collapse spaces
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()


def remove_urls_and_emails(text: str) -> str:
    """Removes raw URLs, emails, and obfuscated emails."""
    text = safe_text(text)

    text = re.sub(r"\[email[^\]]*\]|\bemail\s+protected\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.IGNORECASE)

    return text


def remove_contact_form_text(text: str) -> str:
    """Removes contact/comment form text scraped from websites."""
    text = safe_text(text)

    text = re.sub(
        r"Your name\s+Your email\s+Subject\s+Your message(?:\s+optional)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Save my name,\s*email,\s*and website in this browser for the next time I comment\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_safe_social_cta(text: str) -> str:
    """
    Removes short social media call-to-action boilerplate.
    Conservative: targets CTA phrases, not normal article paragraphs.
    """
    text = safe_text(text)

    text = re.sub(
        r"Warbixinada qotada dheer iyo wararka BBC Somali[^.]{0,220}?WhatsApp\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b(Halkaan kaga soo biir|Dhamaadka xayeysiinta|follow us|subscribe|telegram channel|whatsapp channel)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def clean_headline_max(text: str) -> str:
    """
    Conservative headline cleaning.
    Removes media labels and site-name suffixes only.
    """
    text = normalize_text(text)

    text = re.sub(
        r"^(Sawirro|Sawir|Daawo|DAAWO|Muqaal|VIDEO|PHOTO|Akhriso|AKHRISO|Akhri|Maqal)\s*[:\-–—]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*[|\-–—]\s*(Caasimada(\s+Online)?|Goobjoog(\s+News)?|Mustaqbal(\s+Media)?|"
        r"Kooxda\.com|Kubadlive|Laacibnet|Puntland\s+Post|SONNA|Wararka24|BBC).*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"^[\"“”'‘’]+|[\"“”'‘’]+$", "", text)

    return normalize_text(text)


# ─────────────────────────────────────────────────────────────────────────────
# Website-specific cleaners
# ─────────────────────────────────────────────────────────────────────────────

def remove_bbc_boilerplate(text: str) -> str:
    """
    BBC Somali cleaner.

    Important:
    Removes 'End of content' as a marker only.
    It does NOT remove everything after it because useful article text can continue.
    """
    text = safe_text(text)

    bbc_credits = (
        r"Getty Images?|Reuters|AFP|EPA|AP|PA Media|BBC|FIFA|SONNA|CAFONLINE|Cafonline|"
        r"Instagram|Facebook|X|G\.D\.|Villa|Baraha Bulshada|Social Media|FC Barcelona"
    )

    text = re.sub(
        rf"Xigashada Sawirka\s*,?\s*(?:{bbc_credits})?\s*\.?\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bEnd of content\b|\bEnd of Ugu akhris badan\b|\bUgu akhris badan\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Warbixinada qotada dheer iyo wararka BBC Somali[^.]{0,220}?WhatsApp\.?"
        r"(?:\s*Halkaan kaga soo biir)?\s*(?:Dhamaadka xayeysiinta)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_caasimada_boilerplate(text: str) -> str:
    """Caasimada Online cleaner."""
    text = safe_text(text).strip()

    # Example: Share Muqdisho (Caasimada Online) –
    text = re.sub(
        r"^Share\s+.{0,140}?(?:Caasimada\s+Onl(?:ine|ien)|Caasimada\s+Online\)?)[^–—-]{0,30}[–—-]\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Fallback for malformed Share datelines
    text = re.sub(
        r"^Share\s+[A-Za-zÀ-ÖØ-öø-ÿ\s().,'-]{1,100}\s*[–—-]\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bRead more\b", " ", text, flags=re.IGNORECASE)

    text = re.sub(
        r"Isha\s*:\s*[A-Za-z][A-Za-z\s/&+.-]{1,100}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bW/[QT]\s*:\s*[^.]{0,180}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Hoos\s+ka\s+daawo[^.]*\.?|Halkan\s+ka\s+daawo[^.]*\.?|HALKAN\s+CLICK\s+SII",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Caasimada\s+Online\s+Xafiiska\s+[\w\s]{1,50}\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Caasimada\s+Online,?\s+waa\s+mareeg[^.]{0,300}(?:Mahadsanid\.?)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Caasimada\s+Online\s+waxay\s+leedahay\s+App[^.]{0,300}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Wire-service credits at the end
    text = re.sub(
        r"(?:AP|VOA|AFP|Reuters|BBC)(?:\s*[,+/&]\s*(?:AP|VOA|AFP|Reuters|BBC|Somali|SOMALI))*\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_goobjoog_boilerplate(text: str) -> str:
    """Goobjoog cleaner."""
    text = safe_text(text)

    text = re.sub(
        r"warkaan\s+wixii\s+kusoo?\s+kordha\s+kala\s+soco\s+wararkeena\s+kale",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"wixii\s+ku\s+soo\s+kordha\s+kala\s+soco\s+Goobjoog\.?com",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"(Halkaan\s+hoose\s+ka\s+(?:daawo|dhageyso)|Hoos\s+Ka\s+Dhageyso|HALKA\s+HOOSE\s+KA\s+DHAGEYSO)[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bWariye\s*:\s*[^.]{0,100}\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bGoobjoog\s+News\b\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_mustaqbal_boilerplate(text: str) -> str:
    """Mustaqbal Media cleaner."""
    text = safe_text(text)

    text = re.sub(
        r"La\s+soco\s+wixii\s+kasoo\s+kordha\s+Mustaqbal\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Isha\s*:\s*[A-Za-z][A-Za-z\s/&+.-]{1,100}\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"aragtida\s+.*?kamana\s+turjumayaan\s+aragtida\s+Mustaqbal\s+Media\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bMustaqbal\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_kooxda_boilerplate(text: str) -> str:
    """Kooxda.com cleaner."""
    text = safe_text(text)

    text = re.sub(r"Kooxda\.com\s+ayaa\s+xusaysa[^.]*\.?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"La\s+soco\s+Kooxda\.com[^.]*\.?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Sawirro?\s+la\s+qaaday[^.]*\.?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bKooxda\.com\b\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_kubadlive_boilerplate(text: str) -> str:
    """Kubadlive cleaner."""
    text = safe_text(text)

    text = re.sub(r"\bViews?\s*:\s*\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Halkan\s+ka\s+daawo\s*:-?", " ", text, flags=re.IGNORECASE)

    return text


def remove_laacibnet_boilerplate(text: str) -> str:
    """Laacibnet cleaner."""
    text = safe_text(text)

    text = re.sub(
        r"Save my name,\s*email,\s*and website in this browser for the next time I comment\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bLaacibnet\b\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_puntlandpost_boilerplate(text: str) -> str:
    """Puntland Post cleaner."""
    text = safe_text(text)

    text = re.sub(r"\bPUNTLAND\s+POST\b\s*$", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bSii\s+akhri\b\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_sonna_boilerplate(text: str) -> str:
    """SONNA cleaner."""
    text = safe_text(text)

    text = re.sub(
        r"^[A-ZÀ-ÖØ-Ýa-zà-öø-ÿ’'\-\s]{2,35}\s*\(?F?SONNA\)?\s*[-:–—]*\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bBy\.?\s+[A-Za-zÀ-ÖØ-öø-ÿ/.'\-\s]{2,80}\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_wararka24_boilerplate(text: str) -> str:
    """Wararka24 cleaner."""
    text = safe_text(text)

    text = re.sub(r"^Wararka24\s+\d{1,2}[-/]\d{1,2}[-/]\d{4}\s*", " ", text, flags=re.IGNORECASE)

    text = re.sub(
        r"^[A-ZÀ-ÖØ-Ýa-zà-öø-ÿ’'\-\s]{2,35}\s*[-–—]\s*Wararka24\s*[-–—]?\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"Wararka24\s*[–—-]\s*Xog rasmi ah[^.]*$", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bWararka24\.com\b", " ", text, flags=re.IGNORECASE)

    return text


def remove_garoweonline_boilerplate(text: str) -> str:
    """GaroweOnline cleaner. Most rows in this dataset appear to be listing/cookie pages."""
    text = safe_text(text)

    text = re.sub(r"Copyright\s+©\s+GAROWONLINE\s+All\s+Rights\s+Reserved", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Advertise with GaroweOnline[^.]*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"To manage your cookie choices[^.]*", " ", text, flags=re.IGNORECASE)

    return text


SITE_CLEANERS = {
    "bbc_somali": [remove_bbc_boilerplate],
    "caasimada": [remove_caasimada_boilerplate],
    "goobjoog": [remove_goobjoog_boilerplate],
    "mustaqbalmedia": [remove_mustaqbal_boilerplate],
    "kooxda": [remove_kooxda_boilerplate],
    "kooxdamanta": [],
    "kubadlive": [remove_kubadlive_boilerplate],
    "laacibnet": [remove_laacibnet_boilerplate],
    "puntlandpost": [remove_puntlandpost_boilerplate],
    "sonna": [remove_sonna_boilerplate],
    "wararka24": [remove_wararka24_boilerplate],
    "garoweonline": [remove_garoweonline_boilerplate],
}


def clean_body_max(text: str, site: str = "") -> str:
    """
    Maximum body cleaning pipeline.
    """
    text = normalize_text(text)
    site = safe_text(site).strip().lower()

    for fn in SITE_CLEANERS.get(site, []):
        text = fn(text)

    text = remove_contact_form_text(text)
    text = remove_urls_and_emails(text)
    text = remove_safe_social_cta(text)

    # Remove trailing teaser ellipsis
    text = re.sub(r"\s*[…]+\s*$", "", text)
    text = re.sub(r"\s*\.{3,}\s*$", "", text)

    return normalize_text(text)


# ─────────────────────────────────────────────────────────────────────────────
# Strong non-Somali / English article removal
# ─────────────────────────────────────────────────────────────────────────────

SOMALI_MARKERS = {
    "ayaa", "ayuu", "ayey", "waxaa", "waxuu", "waxay", "waxan", "waxa",
    "oo", "iyo", "ee", "ku", "ka", "la", "uu", "ay",
    "in", "waa", "ah", "ama", "haddii", "sidoo", "kale",
    "lagu", "looga", "ugu", "kaga", "iska", "loogu", "wuxuu", "waxayna",
    "dalka", "magaalada", "dowladda", "shacabka", "caalamka",
    "madaxweynaha", "wasaaradda", "ciidanka", "kooxda", "xukuumadda",
    "gobolka", "degmada", "muqdisho", "soomaaliya", "somaliland",
    "ayaa sheegay", "waxaa uu", "sida uu",
}

# Avoid very ambiguous terms like "in" alone as English evidence.
ENGLISH_MARKERS = {
    "the", "and", "is", "of", "to", "that", "it", "for",
    "with", "on", "are", "was", "has", "have", "this", "an",
    "at", "by", "from", "or", "be", "as", "not", "but", "his",
    "her", "their", "you", "your", "we", "our", "can", "will",
    "would", "should", "privacy", "policy", "terms", "conditions",
    "cookie", "advertise", "copyright", "reserved",
}

ENGLISH_PAGE_PHRASES_RE = re.compile(
    r"privacy policy|terms and conditions|cookie policy|contact us|advertise with|"
    r"all rights reserved|manage your cookie choices|your name\s+your email|"
    r"save my name,\s*email,\s*and website|editorial policy|disclaimer",
    re.IGNORECASE
)


def word_list(text: str):
    return re.findall(r"\b[\w’'-]+\b", safe_text(text).lower())


def marker_ratio_from_words(words, markers: set) -> float:
    if not words:
        return 0.0

    # Count exact single-word markers only.
    single_markers = {m for m in markers if " " not in m}
    return sum(1 for w in words if w in single_markers) / len(words)


def contains_somali_phrase(text: str) -> bool:
    text_l = safe_text(text).lower()
    phrase_markers = [
        "ayaa sheegay",
        "waxaa uu",
        "waxay sheegtay",
        "sida uu",
        "sida ay",
        "isagoo sheegay",
        "kuwaas oo",
        "ka dib markii",
    ]
    return any(p in text_l for p in phrase_markers)


def detect_language_library(text: str):
    """
    Returns:
      langdetect_lang, langid_lang
    Empty string means library not installed or detection failed.
    """
    sample = safe_text(text)[:1500]
    langdetect_lang = ""
    langid_lang = ""

    if detect is not None:
        try:
            langdetect_lang = detect(sample)
        except Exception:
            langdetect_lang = ""

    if langid is not None:
        try:
            langid_lang, _confidence = langid.classify(sample)
        except Exception:
            langid_lang = ""

    return langdetect_lang, langid_lang


def is_somali_article_max(headline: str, body: str, site: str = "") -> tuple:
    """
    Stronger version of the previous notebook's language filter.

    Returns:
      (is_somali: bool, reason: str)

    Main principle:
    - Keep rows with clear Somali evidence.
    - Remove clear English/policy/contact/non-Somali rows.
    - Remove rows with no Somali evidence after cleaning.
    """
    headline = safe_text(headline)
    body = safe_text(body)
    site = safe_text(site).lower()

    text = normalize_text(f"{headline} {body}")
    words = word_list(text)
    word_count = len(words)

    if word_count == 0:
        return False, "empty_text"

    so_ratio = marker_ratio_from_words(words, SOMALI_MARKERS)
    en_ratio = marker_ratio_from_words(words, ENGLISH_MARKERS)
    so_count = sum(1 for w in words if w in {m for m in SOMALI_MARKERS if " " not in m})
    en_count = sum(1 for w in words if w in ENGLISH_MARKERS)
    has_so_phrase = contains_somali_phrase(text)
    has_english_page_phrase = bool(ENGLISH_PAGE_PHRASES_RE.search(text))

    langdetect_lang, langid_lang = detect_language_library(text)

    # Clear non-article English pages
    if has_english_page_phrase and so_ratio < 0.025 and not has_so_phrase:
        return False, (
            f"english_boilerplate_page("
            f"so={so_ratio:.3f},en={en_ratio:.3f},ld={langdetect_lang},li={langid_lang})"
        )

    # Very short rows are removed from final dataset anyway, but mark as non-Somali
    # if they have no Somali evidence.
    if word_count < 30 and so_count < 2 and not has_so_phrase:
        return False, f"too_short_no_somali_evidence(words={word_count},so={so_ratio:.3f})"

    # Strong Somali heuristic
    if (so_ratio >= 0.045 and so_count >= 4) or has_so_phrase:
        # But protect against obvious English policy pages.
        if en_ratio >= 0.18 and so_ratio < 0.035 and not has_so_phrase:
            return False, f"english_heavy_despite_some_somali(so={so_ratio:.3f},en={en_ratio:.3f})"

        return True, f"heuristic_somali(so={so_ratio:.3f},en={en_ratio:.3f})"

    # Library says Somali.
    if langdetect_lang == "so" or langid_lang == "so":
        # Keep only if not strongly English-heavy.
        if en_ratio >= 0.12 and so_ratio < 0.025:
            return False, f"library_so_but_english_heavy(so={so_ratio:.3f},en={en_ratio:.3f})"

        return True, f"library_somali(ld={langdetect_lang},li={langid_lang},so={so_ratio:.3f})"

    # Somali is sometimes misdetected as Swahili/Indonesian/Malay/Tagalog.
    # Keep only if there is at least some Somali marker evidence.
    if langdetect_lang in {"sw", "id", "ms", "tl"} or langid_lang in {"sw", "id", "ms", "tl"}:
        if so_ratio >= 0.020 and so_count >= 2:
            return True, f"near_somali_lang_with_markers(ld={langdetect_lang},li={langid_lang},so={so_ratio:.3f})"

    # Clear English evidence
    if (langdetect_lang == "en" or langid_lang == "en") and en_ratio >= 0.055 and so_ratio < 0.030:
        return False, f"english_detected(ld={langdetect_lang},li={langid_lang},so={so_ratio:.3f},en={en_ratio:.3f})"

    # Marker-only English detection
    if en_ratio >= 0.080 and en_count >= 5 and so_ratio < 0.025:
        return False, f"english_marker_heavy(so={so_ratio:.3f},en={en_ratio:.3f})"

    # No Somali evidence at all.
    if so_count == 0 and not has_so_phrase:
        return False, f"no_somali_evidence(ld={langdetect_lang},li={langid_lang},en={en_ratio:.3f})"

    # Ambiguous but weak Somali evidence: keep for manual review if not clearly English.
    if so_count >= 2 and en_ratio < 0.10:
        return True, f"weak_somali_evidence_kept(so={so_ratio:.3f},en={en_ratio:.3f})"

    return False, f"non_somali_or_unknown(so={so_ratio:.3f},en={en_ratio:.3f},ld={langdetect_lang},li={langid_lang})"


# ─────────────────────────────────────────────────────────────────────────────
# Article-level removal rules
# ─────────────────────────────────────────────────────────────────────────────

POLICY_HEADLINE_RE = re.compile(
    r"^(privacy\s*policy|terms\s+(and\s+conditions|of\s+service)|editorial\s+policy|"
    r"cookie\s+policy|contact\s+us|nala\s+soo\s+xiriir|adverti[sz]e\s+with\s+us|"
    r"bookmark\s+page|disclaimer|guidance:\s*links\s+and\s+feeds)$",
    re.IGNORECASE
)

POLICY_URL_RE = re.compile(
    r"/(privacy|terms|cookie|contact|contactus|nala-soo-xiriir|advertise|advertize|disclaimer|editorial|bookmark)",
    re.IGNORECASE
)

GOOBJOOG_NAV_TITLES = {
    "Wararka Caalamka", "Dhaqanka & Buugta", "Aragtida Xorta Ah",
    "Arrimaha Ganacsiga", "La-dagaallanka Al-Shabaab", "Deegaanka & Cimilada",
    "Arrimaha Amniga", "Dagaalka Dib-u-Xoreynta", "Siyaasadda Xogdhawrka",
    "Wararka Maanta", "Ciyaaraha", "Caafimaad", "Diinta",
}


def get_article_removal_reason(row) -> str:
    """
    Returns a reason if the entire row should be removed.
    Otherwise returns empty string.
    """
    site = safe_text(row.get("site", "")).strip().lower()
    headline = safe_text(row.get("headline_original", "")).strip()
    url = safe_text(row.get("url", "")).strip()
    body = safe_text(row.get("body_original", "")).strip()

    if headline in GOOBJOOG_NAV_TITLES:
        return "navigation_page"

    if POLICY_HEADLINE_RE.search(headline):
        return "policy_or_contact_page"

    if POLICY_URL_RE.search(url):
        return "policy_or_contact_url"

    if re.search(r"Your name\s+Your email\s+Subject", body, re.IGNORECASE):
        return "contact_form_page"

    if site == "garoweonline" and re.search(
        r"Advertise with GaroweOnline|manage your cookie choices|Copyright © GAROWONLINE",
        body,
        re.IGNORECASE
    ):
        return "garoweonline_listing_or_cookie_page"

    if re.search(
        r"Xuquuqda Gaarka ah ee CCPA|Ha Iibiina Macluumaadkayga",
        body,
        re.IGNORECASE
    ):
        return "privacy_ccpa_page"

    return ""


def normalized_hash(text: str) -> str:
    text = normalize_text(text).lower()
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def headline_inside_body(row) -> bool:
    headline = normalize_text(row.get("headline_clean", "")).lower()
    body = normalize_text(row.get("body_clean", "")).lower()

    if len(headline) < 20:
        return False

    return headline in body[:600]


def make_suspicious_flags(row) -> list:
    """
    Flags rows for review.
    Flags do not always mean deletion; they help manual inspection.
    """
    flags = []

    body_clean = safe_text(row.get("body_clean", ""))
    headline_clean = safe_text(row.get("headline_clean", ""))
    body_words = int(row.get("body_words_after", 0))
    link_count = len(re.findall(r"https?://|www\.", safe_text(row.get("body_original", ""))))

    if row.get("removal_reason", ""):
        flags.append(f"remove:{row.get('removal_reason')}")

    if row.get("language_reason", ""):
        flags.append(f"language:{row.get('language_reason')}")

    if not headline_clean.strip():
        flags.append("empty_headline_after_cleaning")

    if not body_clean.strip():
        flags.append("empty_body_after_cleaning")

    if len(headline_clean) < MIN_HEADLINE_CHARS:
        flags.append("very_short_headline")

    if len(headline_clean) > MAX_HEADLINE_CHARS:
        flags.append("very_long_headline")

    if body_words < MIN_BODY_WORDS_FINAL:
        flags.append(f"very_short_body({body_words}_words)")

    if body_words > MAX_BODY_WORDS_REVIEW:
        flags.append(f"very_long_body({body_words}_words)")

    if row.get("cleaning_loss_ratio", 0) > 0.65:
        flags.append(f"cleaning_removed_too_much({row.get('cleaning_loss_ratio'):.0%})")

    if row.get("duplicate_url", False):
        flags.append("duplicate_url")

    if row.get("duplicate_clean_body", False):
        flags.append("duplicate_clean_body")

    if link_count > 2:
        flags.append(f"too_many_links({link_count})")

    if headline_inside_body(row):
        flags.append("headline_repeated_inside_body")

    residual_boilerplate = re.search(
        r"Read more|Xigashada Sawirka|Goobjoog News|PUNTLAND POST|Save my name|Views?\s*:\s*\d+|"
        r"Advertise with GaroweOnline|manage your cookie choices|Your name\s+Your email",
        body_clean,
        re.IGNORECASE
    )

    if residual_boilerplate:
        flags.append("residual_boilerplate_after_cleaning")

    if safe_text(row.get("category", "")) not in VALID_CATEGORIES:
        flags.append(f"unexpected_category({row.get('category', '')})")

    return flags


print("✅ Maximum cleaning functions loaded.")
print(f"Configured site cleaners: {list(SITE_CLEANERS.keys())}")
print(f"langdetect available: {detect is not None}")
print(f"langid available: {langid is not None}")


✅ Maximum cleaning functions loaded.
Configured site cleaners: ['bbc_somali', 'caasimada', 'goobjoog', 'mustaqbalmedia', 'kooxda', 'kooxdamanta', 'kubadlive', 'laacibnet', 'puntlandpost', 'sonna', 'wararka24', 'garoweonline']
langdetect available: True
langid available: True


In [5]:

# CELL 5 — APPLY MAXIMUM CLEANING + REMOVE NON-SOMALI ARTICLES
# ============================================================

work_df = raw_df.copy()

# Ensure expected columns exist
for col in ["site", "category", "url", "headline", "body", "source_file"]:
    if col not in work_df.columns:
        work_df[col] = ""

work_df["site"] = work_df["site"].astype(str).str.strip().str.lower()
work_df["category"] = work_df["category"].astype(str).str.strip().str.lower()

# Keep originals for inspection
work_df["headline_original"] = work_df["headline"].astype(str)
work_df["body_original"] = work_df["body"].astype(str)

print("🧹 Cleaning headlines...")
work_df["headline_clean"] = work_df["headline_original"].apply(clean_headline_max)

print("🧹 Cleaning bodies...")
work_df["body_clean"] = work_df.apply(
    lambda row: clean_body_max(row["body_original"], row["site"]),
    axis=1
)

# Length columns
work_df["headline_chars_before"] = work_df["headline_original"].str.len()
work_df["headline_chars_after"] = work_df["headline_clean"].str.len()

work_df["body_chars_before"] = work_df["body_original"].str.len()
work_df["body_chars_after"] = work_df["body_clean"].str.len()

work_df["body_words_before"] = work_df["body_original"].str.split().str.len()
work_df["body_words_after"] = work_df["body_clean"].str.split().str.len()

work_df["cleaning_loss_ratio"] = 1 - (
    work_df["body_chars_after"] / work_df["body_chars_before"].replace(0, np.nan)
)

work_df["cleaning_loss_ratio"] = work_df["cleaning_loss_ratio"].fillna(0)

# Article-level removal reasons
print("🔎 Marking clear non-article rows...")
work_df["removal_reason"] = work_df.apply(get_article_removal_reason, axis=1)

# Strong language detection on cleaned text
print("🌍 Removing English / non-Somali articles using strong language checks...")
lang_results = work_df.apply(
    lambda row: is_somali_article_max(
        row["headline_clean"],
        row["body_clean"],
        row["site"]
    ),
    axis=1
)

work_df["_is_somali"], work_df["language_reason"] = zip(*lang_results)

# Mark non-Somali rows for removal
work_df.loc[
    work_df["removal_reason"].eq("") & ~work_df["_is_somali"],
    "removal_reason"
] = "non_somali_article"

# Duplicate helpers
work_df["url_norm"] = work_df["url"].astype(str).str.strip().str.lower()
work_df["body_hash"] = work_df["body_clean"].apply(normalized_hash)

work_df["duplicate_url"] = (
    work_df["url_norm"].ne("") &
    work_df.duplicated("url_norm", keep=False)
)

work_df["duplicate_clean_body"] = (
    work_df["body_clean"].str.len().gt(50) &
    work_df.duplicated("body_hash", keep=False)
)

# Suspicious flags
print("🚩 Creating suspicious flags...")
work_df["suspicious_flags"] = work_df.apply(make_suspicious_flags, axis=1)
work_df["suspicious_reason"] = work_df["suspicious_flags"].apply(lambda flags: "; ".join(flags))
work_df["is_suspicious"] = work_df["suspicious_flags"].apply(lambda flags: len(flags) > 0)

# Build removed datasets
removed_rows_all_reasons = work_df[work_df["removal_reason"].ne("")].copy()
removed_non_somali_articles = work_df[work_df["removal_reason"].eq("non_somali_article")].copy()
suspicious_rows_for_review = work_df[work_df["is_suspicious"]].copy()

# Build final cleaned dataset
cleaned_dataset = work_df.copy()

# Remove non-article and non-Somali rows
cleaned_dataset = cleaned_dataset[cleaned_dataset["removal_reason"].eq("")].copy()

# Remove empty cleaned values
cleaned_dataset = cleaned_dataset[
    cleaned_dataset["headline_clean"].str.strip().ne("") &
    cleaned_dataset["body_clean"].str.strip().ne("")
].copy()

# Remove very short bodies from final dataset
# They remain available in suspicious_rows_for_review.
cleaned_dataset = cleaned_dataset[cleaned_dataset["body_words_after"] >= MIN_BODY_WORDS_FINAL].copy()

# Deduplicate by URL, but do not collapse empty URLs together
with_url = cleaned_dataset[cleaned_dataset["url_norm"].ne("")].drop_duplicates("url_norm", keep="first")
without_url = cleaned_dataset[cleaned_dataset["url_norm"].eq("")]
cleaned_dataset = pd.concat([with_url, without_url], ignore_index=True)

# Deduplicate by cleaned body hash
cleaned_dataset = cleaned_dataset.drop_duplicates("body_hash", keep="first").copy()

# Final training-friendly columns
cleaned_dataset["headline"] = cleaned_dataset["headline_clean"]
cleaned_dataset["body"] = cleaned_dataset["body_clean"]

# Add stable IDs
cleaned_dataset = cleaned_dataset.reset_index(drop=True)
cleaned_dataset["id"] = [
    f"somali_news_cleaned_{i+1:05d}"
    for i in range(len(cleaned_dataset))
]

# Backward compatibility with older notebook cells
df = cleaned_dataset.copy()

print("\n" + "═" * 90)
print("✅ MAXIMUM CLEANING COMPLETE")
print("═" * 90)
print(f"Raw rows: {len(raw_df):,}")
print(f"Rows removed for all reasons: {len(removed_rows_all_reasons):,}")
print(f"Non-Somali / English rows removed: {len(removed_non_somali_articles):,}")
print(f"Rows after cleaning/removal/deduplication: {len(cleaned_dataset):,}")
print(f"Suspicious rows for review: {len(suspicious_rows_for_review):,}")

print("\n📌 Removal reasons:")
display(
    work_df["removal_reason"]
    .replace("", "kept")
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="rows")
)

print("\n📌 Language decision examples:")
display(
    work_df["language_reason"]
    .value_counts()
    .head(20)
    .rename_axis("language_reason")
    .reset_index(name="rows")
)

print("\n📌 Cleaned rows by source:")
display(
    cleaned_dataset["site"]
    .value_counts()
    .rename_axis("site")
    .reset_index(name="rows")
)

print("\n📌 Cleaned rows by category:")
display(
    cleaned_dataset["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

print("\n📌 Preview cleaned dataset:")
display(cleaned_dataset[["id", "site", "category", "headline", "body"]].head())


🧹 Cleaning headlines...
🧹 Cleaning bodies...
🔎 Marking clear non-article rows...
🌍 Removing English / non-Somali articles using strong language checks...
🚩 Creating suspicious flags...

══════════════════════════════════════════════════════════════════════════════════════════
✅ MAXIMUM CLEANING COMPLETE
══════════════════════════════════════════════════════════════════════════════════════════
Raw rows: 35,797
Rows removed for all reasons: 127
Non-Somali / English rows removed: 75
Rows after cleaning/removal/deduplication: 34,477
Suspicious rows for review: 35,797

📌 Removal reasons:


,reason,rows
0,kept,35670
1,non_somali_article,75
2,navigation_page,19
3,policy_or_contact_page,18
4,garoweonline_listing_or_cookie_page,10
5,policy_or_contact_url,5



📌 Language decision examples:


,language_reason,rows
0,"heuristic_somali(so=0.333,en=0.000)",348
1,"heuristic_somali(so=0.286,en=0.000)",341
2,"heuristic_somali(so=0.291,en=0.000)",324
3,"heuristic_somali(so=0.312,en=0.000)",322
4,"heuristic_somali(so=0.297,en=0.000)",320
5,"heuristic_somali(so=0.293,en=0.000)",314
6,"heuristic_somali(so=0.299,en=0.000)",312
7,"heuristic_somali(so=0.289,en=0.000)",308
8,"heuristic_somali(so=0.287,en=0.000)",307
9,"heuristic_somali(so=0.283,en=0.000)",305



📌 Cleaned rows by source:


,site,rows
0,goobjoog,12339
1,caasimada,8753
2,kooxda,7165
3,mustaqbalmedia,4562
4,laacibnet,1048
5,bbc_somali,234
6,puntlandpost,222
7,kubadlive,109
8,sonna,24
9,wararka24,13



📌 Cleaned rows by category:


,category,rows
0,caalamka,10008
1,siyaasad,9301
2,ciyaaro,8898
3,amni,6270



📌 Preview cleaned dataset:


,id,site,category,headline,body
0,somali_news_cleaned_00001,bbc_somali,ciyaaro,Maxay dalalku ka faa'idaan marka ay u soo baxa...,Dalalka si guul leh ugu soo baxa Koobka Adduun...
1,somali_news_cleaned_00002,bbc_somali,ciyaaro,Maxaa ka jira in ciyaartooyda kooxdii caanka a...,Kooxda Real Madrid ayaa toddobaadkan gashay xa...
2,somali_news_cleaned_00003,bbc_somali,ciyaaro,Ciyaaryahanada ugu sarreeya Afrika ee aan ka q...,Iyada oo ay soo dhowaatay isku-aadka Koobka Ad...
3,somali_news_cleaned_00004,bbc_somali,ciyaaro,Maxaa ugu wacan Arsenal inaysan weligeed ku gu...,Arsenal waligeed kuma guuleysan Champions Leag...
4,somali_news_cleaned_00005,bbc_somali,ciyaaro,Dhamaan xogta iyo macluumaadka aad u baahan ta...,Iyadoo dhamaan kooxaha ka qayb galaya koobka a...


In [6]:

# CELL 6 — INSPECT REMOVED ENGLISH / NON-SOMALI ARTICLES
# ======================================================

print(f"Removed non-Somali / English rows: {len(removed_non_somali_articles):,}")

if len(removed_non_somali_articles) > 0:
    preview_cols = [
        "site",
        "category",
        "url",
        "headline_original",
        "headline_clean",
        "body_words_after",
        "language_reason",
        "body_original",
        "body_clean",
    ]

    preview_cols = [c for c in preview_cols if c in removed_non_somali_articles.columns]

    display(
        removed_non_somali_articles[preview_cols]
        .sample(min(20, len(removed_non_somali_articles)), random_state=42)
    )
else:
    print("✅ No non-Somali articles were removed.")


Removed non-Somali / English rows: 75


,site,category,url,headline_original,headline_clean,body_words_after,language_reason,body_original,body_clean
13010,goobjoog,amni,https://goobjoog.com/2021/05/14/ogeysiis-muhii...,OGEYSIIS MUHIIM AH iyo UN CAREER FAIR EE MAALM...,OGEYSIIS MUHIIM AH iyo UN CAREER FAIR EE MAALM...,121,"non_somali_or_unknown(so=0.030,en=0.303,ld=en,...",The UN Somalia Human Resources team will host ...,The UN Somalia Human Resources team will host ...
30876,laacibnet,ciyaaro,https://www.laacibnet.net/geekom-geekbook-x14-...,GEEKOM GeekBook X14 Pro 14″ OLED Ultra Thin La...,GEEKOM GeekBook X14 Pro 14″ OLED Ultra Thin La...,500,"english_detected(ld=en,li=en,so=0.004,en=0.200)",Honest GEEKOM GeekBook X14 Pro Laptop evaluate...,Honest GEEKOM GeekBook X14 Pro Laptop evaluate...
18324,goobjoog,siyaasad,https://goobjoog.com/2022/03/21/hormuuds-evc-m...,Hormuud’s EVC mobile money service celebrates ...,Hormuud’s EVC mobile money service celebrates ...,895,"english_detected(ld=en,li=en,so=0.030,en=0.274)",Hormuud Telecom’s EVC Plus service this week c...,Hormuud Telecom’s EVC Plus service this week c...
24,bbc_somali,ciyaaro,https://www.bbc.com/ws/languages,"BBC News , World Service - Home","BBC News, World Service - Home",114,"english_detected(ld=en,li=en,so=0.017,en=0.200)",BBC News is the world’s most trusted internati...,BBC News is the world’s most trusted internati...
30848,laacibnet,ciyaaro,https://www.laacibnet.net/thomas-frank-hails-b...,Thomas Frank Hails ‘Big Step’ as Tottenham’s E...,Thomas Frank Hails ‘Big Step’ as Tottenham’s E...,841,"non_somali_or_unknown(so=0.036,en=0.261,ld=en,...",Thomas Frank praised Tottenham’s “first-rate” ...,Thomas Frank praised Tottenham’s “first-rate” ...
30874,laacibnet,ciyaaro,https://www.laacibnet.net/google-pixel-10-best...,Google Pixel 10 Best Deals Specs & Reviews,Google Pixel 10 Best Deals Specs & Reviews,519,"english_detected(ld=en,li=en,so=0.013,en=0.190)",The Google Pixel 10 is Google’s flagship phone...,The Google Pixel 10 is Google’s flagship phone...
22547,kooxda,ciyaaro,https://kooxda.com/list/chicken-road-funny-chi...,Chicken Road - Online Casino Slot Featuring Fu...,Chicken Road - Online Casino Slot Featuring Fu...,1959,"english_detected(ld=en,li=en,so=0.012,en=0.326)",Content chicken road is a unique and entertain...,Content chicken road is a unique and entertain...
22503,kooxda,ciyaaro,https://kooxda.com/list/chicken-road-slot-unen...,Chicken Road - Online Casino Slot Offering End...,Chicken Road - Online Casino Slot Offering End...,1386,"english_detected(ld=en,li=en,so=0.012,en=0.289)",Content Discover the thrilling world of Chicke...,Content Discover the thrilling world of Chicke...
30882,laacibnet,ciyaaro,https://www.laacibnet.net/320k-year-business-s...,320K Year Business Start Tomorrow With 400,320K Year Business Start Tomorrow With 400,1565,"english_detected(ld=en,li=en,so=0.014,en=0.234)",You can begin a home made cleaning soap busine...,You can begin a home made cleaning soap busine...
30877,laacibnet,ciyaaro,https://www.laacibnet.net/14%e2%80%b3-laptop-c...,14″ Laptop Computer with Gold 6500Y – 16GB RAM...,14″ Laptop Computer with Gold 6500Y – 16GB RAM...,607,"english_detected(ld=en,li=en,so=0.010,en=0.201)","Laptop Computer with Gold 6500Y processor, 16G...","Laptop Computer with Gold 6500Y processor, 16G..."


In [7]:

# CELL 7 — BEFORE / AFTER INSPECTION
# ==================================

def get_removed_chunks(original: str, cleaned: str, max_chunks: int = 6) -> str:
    """
    Shows text chunks that disappeared after cleaning.
    This is only for inspection.
    """
    original = normalize_text(original)
    cleaned = normalize_text(cleaned)

    matcher = SequenceMatcher(None, original, cleaned)
    removed = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ["delete", "replace"]:
            chunk = original[i1:i2].strip()

            if len(chunk) >= 20:
                removed.append(chunk)

        if len(removed) >= max_chunks:
            break

    return " | ".join(removed)


def inspect_cleaning(site=None, category=None, suspicious_only=False, removed_only=False, n=10, random_state=42):
    """
    Shows before/after cleaning examples.

    Usage examples:
      inspect_cleaning(site="bbc_somali", n=5)
      inspect_cleaning(site="caasimada", suspicious_only=True, n=10)
      inspect_cleaning(removed_only=True, n=20)
      inspect_cleaning(category="ciyaaro", n=10)
    """
    temp = work_df.copy()

    if site is not None:
        temp = temp[temp["site"].eq(site)]

    if category is not None:
        temp = temp[temp["category"].eq(category)]

    if suspicious_only:
        temp = temp[temp["is_suspicious"]]

    if removed_only:
        temp = temp[temp["removal_reason"].ne("")]

    if temp.empty:
        print("No rows found for this filter.")
        return

    sample = temp.sample(min(n, len(temp)), random_state=random_state).copy()

    sample["removed_text_sample"] = sample.apply(
        lambda row: get_removed_chunks(row["body_original"], row["body_clean"]),
        axis=1
    )

    cols = [
        "site",
        "category",
        "removal_reason",
        "language_reason",
        "headline_original",
        "headline_clean",
        "body_words_before",
        "body_words_after",
        "cleaning_loss_ratio",
        "suspicious_reason",
        "removed_text_sample",
        "body_original",
        "body_clean",
    ]

    cols = [c for c in cols if c in sample.columns]

    display(sample[cols])


# Example inspections
inspect_cleaning(site="bbc_somali", n=3)
inspect_cleaning(site="caasimada", n=3)
inspect_cleaning(removed_only=True, n=5)


,site,category,removal_reason,language_reason,headline_original,headline_clean,body_words_before,body_words_after,cleaning_loss_ratio,suspicious_reason,removed_text_sample,body_original,body_clean
24,bbc_somali,ciyaaro,non_somali_article,"english_detected(ld=en,li=en,so=0.017,en=0.200)","BBC News , World Service - Home","BBC News, World Service - Home",114,114,0.000000,remove:non_somali_article; language:english_de...,,BBC News is the world’s most trusted internati...,BBC News is the world’s most trusted internati...
6,bbc_somali,ciyaaro,,"heuristic_somali(so=0.279,en=0.000)",Maxay kala kulmeen Australia ciyaartoydii reer...,Maxay kala kulmeen Australia ciyaartoydii reer...,419,391,0.063986,"language:heuristic_somali(so=0.279,en=0.000)",End of Ugu akhris badan End of content Warbixi...,"Xigashada Sawirka, jack tan Laba ciyaartoy oo ...",jack tan Laba ciyaartoy oo kubadda cagta ah oo...
153,bbc_somali,ciyaaro,,"heuristic_somali(so=0.248,en=0.009)",Weeraryahanka Kenya ku dhashay ee Soomaaliya u...,Weeraryahanka Kenya ku dhashay ee Soomaaliya u...,471,441,0.065098,"language:heuristic_somali(so=0.248,en=0.009)",End of Ugu akhris badan | End of content Xigas...,"Xigashada Sawirka, Syracuse Orange Weerar yaha...",Syracuse Orange Weerar yahanka Kenya ku dhasha...


,site,category,removal_reason,language_reason,headline_original,headline_clean,body_words_before,body_words_after,cleaning_loss_ratio,suspicious_reason,removed_text_sample,body_original,body_clean
2036,caasimada,caalamka,,"heuristic_somali(so=0.290,en=0.002)",Doorasho maanta ka socota dalka Jabuuti xilli ...,Doorasho maanta ka socota dalka Jabuuti xilli ...,488,479,0.017567,"language:heuristic_somali(so=0.290,en=0.002)",Share Jabuuti (Caasimada Online) –,Share Jabuuti (Caasimada Online) – Cod-bixiyay...,Cod-bixiyayaasha Jabuuti ayaa maanta dooranaya...
4129,caasimada,caalamka,,"heuristic_somali(so=0.211,en=0.000)",Xiisadda Eritrea iyo Jabuuti oo cirka isku sha...,Xiisadda Eritrea iyo Jabuuti oo cirka isku sha...,166,157,0.056401,"language:heuristic_somali(so=0.211,en=0.000)",Share Jabuuti (Caasimada Online) – | [email pr...,Share Jabuuti (Caasimada Online) – Midowga Afr...,Midowga Afrika ayaa ku baxay xiisada ka dhex a...
8922,caasimada,siyaasad,,"heuristic_somali(so=0.266,en=0.005)","Daawo: Moole, Joojole, Jeex iyo Banyaa oo la h...","Moole, Joojole, Jeex iyo Banyaa oo la horgeeya...",388,381,0.018563,"language:heuristic_somali(so=0.266,en=0.005)",Share Muqdisho (Caasimada Online) –,Share Muqdisho (Caasimada Online) – Maxkamadda...,Maxkamadda Ciidamada Qalaba Sida ayaa waxaa ma...


,site,category,removal_reason,language_reason,headline_original,headline_clean,body_words_before,body_words_after,cleaning_loss_ratio,suspicious_reason,removed_text_sample,body_original,body_clean
9624,goobjoog,amni,navigation_page,"heuristic_somali(so=0.346,en=0.000)",Arrimaha Ganacsiga,Arrimaha Ganacsiga,299,299,0.000000,remove:navigation_page; language:heuristic_som...,,Waxaa maanta gabi ahaanba xiran oo aan shaqayn...,Waxaa maanta gabi ahaanba xiran oo aan shaqayn...
30880,laacibnet,ciyaaro,non_somali_article,"english_detected(ld=en,li=en,so=0.020,en=0.225)",How I Bought A $575K Duplex At Age 27 By Myself,How I Bought A $575K Duplex At Age 27 By Myself,1660,1645,0.007216,remove:non_somali_article; language:english_de...,"Save my name, email, and website in this brows...","Margaret Skiff, 27, sold a $575,000 duplex in ...","Margaret Skiff, 27, sold a $575,000 duplex in ..."
30850,laacibnet,ciyaaro,non_somali_article,"english_detected(ld=en,li=en,so=0.028,en=0.248)",Infantino joke about British fans was ‘cheap’ ...,Infantino joke about British fans was ‘cheap’ ...,394,378,0.032903,remove:non_somali_article; language:english_de...,"Save my name, email, and website in this brows...",Fifa president Gianni Infantino should cogniza...,Fifa president Gianni Infantino should cogniza...
22511,kooxda,ciyaaro,non_somali_article,"english_detected(ld=en,li=en,so=0.010,en=0.330)",Glory Casino Login,Glory Casino Login,1880,1880,0.000000,remove:non_somali_article; language:english_de...,,Content Glory Casino APK and Glory Casino App ...,Content Glory Casino APK and Glory Casino App ...
18218,goobjoog,siyaasad,non_somali_article,"non_somali_or_unknown(so=0.037,en=0.300,ld=en,...",Salaam African Bank exports to UGANDA,Salaam African Bank exports to UGANDA,156,154,0.014184,remove:non_somali_article; language:non_somali...,,"Today, we celebrate Salaam African Bank’s mete...","Today, we celebrate Salaam African Bank’s mete..."


In [8]:

# CELL 8 — EXTRA QUALITY CHECKS AFTER MAXIMUM CLEANING
# ====================================================

final_report = cleaned_dataset.copy()

final_report["body_words"] = final_report["body"].astype(str).str.split().str.len()
final_report["headline_chars"] = final_report["headline"].astype(str).str.len()

print("═" * 90)
print("FINAL CLEANED DATASET QUALITY CHECK")
print("═" * 90)

checks = {
    "final_rows": len(final_report),
    "empty_headline": int(final_report["headline"].astype(str).str.strip().eq("").sum()),
    "empty_body": int(final_report["body"].astype(str).str.strip().eq("").sum()),
    "duplicate_urls": int(final_report[final_report["url"].astype(str).str.strip().ne("")].duplicated("url").sum()),
    "duplicate_bodies": int(final_report.duplicated("body").sum()),
    "very_short_bodies_under_50_words": int((final_report["body_words"] < 50).sum()),
    "very_long_bodies_over_1200_words": int((final_report["body_words"] > 1200).sum()),
}

display(pd.DataFrame([checks]).T.rename(columns={0: "count"}))

print("\n📌 Final source distribution:")
display(final_report["site"].value_counts().rename_axis("site").reset_index(name="rows"))

print("\n📌 Final category distribution:")
display(final_report["category"].value_counts().rename_axis("category").reset_index(name="rows"))

print("\n📌 Final body length by source:")
display(
    final_report
    .groupby("site")["body_words"]
    .describe()
    .round(1)
    .reset_index()
)

# Check if English-like rows still remain
print("\n📌 Possible English leftovers in final cleaned dataset:")
possible_english_leftovers = final_report[
    final_report["body"].astype(str).str.contains(
        r"privacy policy|terms and conditions|your name\s+your email|all rights reserved|"
        r"manage your cookie choices|save my name,\s*email,\s*and website",
        case=False,
        regex=True
    )
].copy()

print(f"Possible English/boilerplate leftovers: {len(possible_english_leftovers):,}")

if len(possible_english_leftovers) > 0:
    display(
        possible_english_leftovers[
            ["site", "category", "url", "headline", "body"]
        ].head(20)
    )
else:
    print("✅ No obvious English policy/contact boilerplate found in final dataset.")


══════════════════════════════════════════════════════════════════════════════════════════
FINAL CLEANED DATASET QUALITY CHECK
══════════════════════════════════════════════════════════════════════════════════════════


,count
final_rows,34477
empty_headline,0
empty_body,0
duplicate_urls,0
duplicate_bodies,0
very_short_bodies_under_50_words,0
very_long_bodies_over_1200_words,145



📌 Final source distribution:


,site,rows
0,goobjoog,12339
1,caasimada,8753
2,kooxda,7165
3,mustaqbalmedia,4562
4,laacibnet,1048
5,bbc_somali,234
6,puntlandpost,222
7,kubadlive,109
8,sonna,24
9,wararka24,13



📌 Final category distribution:


,category,rows
0,caalamka,10008
1,siyaasad,9301
2,ciyaaro,8898
3,amni,6270



📌 Final body length by source:


,site,count,mean,std,min,25%,50%,75%,max
0,bbc_somali,234.0,653.8,270.8,74.0,487.8,618.5,803.2,1892.0
1,caasimada,8753.0,361.3,212.2,60.0,237.0,295.0,403.0,2670.0
2,goobjoog,12339.0,178.8,152.6,50.0,102.0,138.0,198.0,2378.0
3,kooxda,7165.0,167.4,98.0,50.0,113.0,145.0,190.0,2342.0
4,kooxdamanta,8.0,327.0,84.0,140.0,320.2,340.5,373.2,417.0
5,kubadlive,109.0,291.9,156.0,63.0,169.0,257.0,370.0,935.0
6,laacibnet,1048.0,175.3,42.9,82.0,145.0,168.0,199.0,421.0
7,mustaqbalmedia,4562.0,216.3,110.9,69.0,160.0,190.5,237.0,1653.0
8,puntlandpost,222.0,201.7,95.7,53.0,138.0,187.0,237.5,574.0
9,sonna,24.0,308.0,258.8,84.0,136.2,202.5,354.8,940.0



📌 Possible English leftovers in final cleaned dataset:
Possible English/boilerplate leftovers: 1


,site,category,url,headline,body
10395,goobjoog,amni,https://goobjoog.com/2022/09/25/hayadda-isgaar...,Hay’adda Isgaarsiinta Qaranka oo Muqdisho Kuso...,Kulankii 11aad ee Hay’adda Maamulka (Board of ...


In [9]:

# CELL 9 — EXPORT MAXIMUM CLEANED DATASET
# =======================================
# CSV files are always saved.
# Excel files are saved only if openpyxl is installed.

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Final cleaned dataset ────────────────────────────────────────────────────
cleaned_export_cols = [
    "id",
    "site",
    "category",
    "url",
    "headline",
    "body",
    "headline_clean",
    "body_clean",
    "body_words_after",
    "language_reason",
    "source_file",
]

cleaned_export_cols = [c for c in cleaned_export_cols if c in cleaned_dataset.columns]
cleaned_export = cleaned_dataset[cleaned_export_cols].copy()

# ── Suspicious review file ───────────────────────────────────────────────────
review_cols = [
    "id",
    "site",
    "category",
    "url",
    "headline_original",
    "headline_clean",
    "body_words_before",
    "body_words_after",
    "cleaning_loss_ratio",
    "removal_reason",
    "language_reason",
    "suspicious_reason",
    "body_original",
    "body_clean",
    "source_file",
]

review_cols = [c for c in review_cols if c in suspicious_rows_for_review.columns]
suspicious_export = suspicious_rows_for_review[review_cols].copy()

# ── Removed non-Somali file ──────────────────────────────────────────────────
removed_non_somali_export = removed_non_somali_articles[
    [c for c in review_cols if c in removed_non_somali_articles.columns]
].copy()

# ── Removed all reasons file ─────────────────────────────────────────────────
removed_all_export = removed_rows_all_reasons[
    [c for c in review_cols if c in removed_rows_all_reasons.columns]
].copy()

# ── Save CSV files ───────────────────────────────────────────────────────────
cleaned_csv_path = OUTPUT_DIR / "cleaned_dataset_maximum.csv"
suspicious_csv_path = OUTPUT_DIR / "suspicious_rows_for_review.csv"
removed_non_somali_csv_path = OUTPUT_DIR / "removed_non_somali_articles.csv"
removed_all_csv_path = OUTPUT_DIR / "removed_rows_all_reasons.csv"

cleaned_export.to_csv(cleaned_csv_path, index=False, encoding="utf-8-sig")
suspicious_export.to_csv(suspicious_csv_path, index=False, encoding="utf-8-sig")
removed_non_somali_export.to_csv(removed_non_somali_csv_path, index=False, encoding="utf-8-sig")
removed_all_export.to_csv(removed_all_csv_path, index=False, encoding="utf-8-sig")

# ── Save Excel files only if openpyxl is installed ───────────────────────────
openpyxl_available = importlib.util.find_spec("openpyxl") is not None

if openpyxl_available:
    cleaned_excel_path = OUTPUT_DIR / "cleaned_dataset_maximum.xlsx"
    suspicious_excel_path = OUTPUT_DIR / "suspicious_rows_for_review.xlsx"
    removed_non_somali_excel_path = OUTPUT_DIR / "removed_non_somali_articles.xlsx"
    removed_all_excel_path = OUTPUT_DIR / "removed_rows_all_reasons.xlsx"

    cleaned_export.to_excel(cleaned_excel_path, index=False, engine="openpyxl")
    suspicious_export.to_excel(suspicious_excel_path, index=False, engine="openpyxl")
    removed_non_somali_export.to_excel(removed_non_somali_excel_path, index=False, engine="openpyxl")
    removed_all_export.to_excel(removed_all_excel_path, index=False, engine="openpyxl")

    print("✅ CSV and Excel files exported successfully.")
    print(f"📁 Cleaned Excel: {cleaned_excel_path.resolve()}")
    print(f"📁 Suspicious Excel: {suspicious_excel_path.resolve()}")
    print(f"📁 Removed non-Somali Excel: {removed_non_somali_excel_path.resolve()}")
    print(f"📁 Removed all reasons Excel: {removed_all_excel_path.resolve()}")
else:
    print("✅ CSV files exported successfully.")
    print("⚠️ Excel files skipped because openpyxl is not installed.")
    print("To enable Excel export, run: pip install openpyxl")

print()
print(f"📁 Cleaned CSV: {cleaned_csv_path.resolve()}")
print(f"📁 Suspicious CSV: {suspicious_csv_path.resolve()}")
print(f"📁 Removed non-Somali CSV: {removed_non_somali_csv_path.resolve()}")
print(f"📁 Removed all reasons CSV: {removed_all_csv_path.resolve()}")

print("\n📊 Final export summary:")
print(f"Cleaned rows: {len(cleaned_export):,}")
print(f"Suspicious review rows: {len(suspicious_export):,}")
print(f"Removed non-Somali rows: {len(removed_non_somali_export):,}")
print(f"Removed rows all reasons: {len(removed_all_export):,}")


✅ CSV and Excel files exported successfully.
📁 Cleaned Excel: E:\somali_cleaner\data\processed\cleaned_dataset_maximum.xlsx
📁 Suspicious Excel: E:\somali_cleaner\data\processed\suspicious_rows_for_review.xlsx
📁 Removed non-Somali Excel: E:\somali_cleaner\data\processed\removed_non_somali_articles.xlsx
📁 Removed all reasons Excel: E:\somali_cleaner\data\processed\removed_rows_all_reasons.xlsx

📁 Cleaned CSV: E:\somali_cleaner\data\processed\cleaned_dataset_maximum.csv
📁 Suspicious CSV: E:\somali_cleaner\data\processed\suspicious_rows_for_review.csv
📁 Removed non-Somali CSV: E:\somali_cleaner\data\processed\removed_non_somali_articles.csv
📁 Removed all reasons CSV: E:\somali_cleaner\data\processed\removed_rows_all_reasons.csv

📊 Final export summary:
Cleaned rows: 34,477
Suspicious review rows: 35,797
Removed non-Somali rows: 75
Removed rows all reasons: 127


In [10]:

# CELL 10 — OPTIONAL: EXPORT TRAINING-READY SIMPLE DATASET
# =======================================================
# This file contains only the columns usually needed for model training.

training_cols = ["id", "category", "headline", "body", "site", "url"]
training_cols = [c for c in training_cols if c in cleaned_dataset.columns]

training_ready = cleaned_dataset[training_cols].copy()

training_csv_path = Path("data/processed/training_ready_cleaned_dataset.csv")
training_ready.to_csv(training_csv_path, index=False, encoding="utf-8-sig")

print("✅ Training-ready dataset exported.")
print(f"📁 Training-ready CSV: {training_csv_path.resolve()}")
print(f"Rows: {len(training_ready):,}")
display(training_ready.head())


✅ Training-ready dataset exported.
📁 Training-ready CSV: E:\somali_cleaner\data\processed\training_ready_cleaned_dataset.csv
Rows: 34,477


,id,category,headline,body,site,url
0,somali_news_cleaned_00001,ciyaaro,Maxay dalalku ka faa'idaan marka ay u soo baxa...,Dalalka si guul leh ugu soo baxa Koobka Adduun...,bbc_somali,https://www.bbc.com/somali/articles/clyp28rnzw0o
1,somali_news_cleaned_00002,ciyaaro,Maxaa ka jira in ciyaartooyda kooxdii caanka a...,Kooxda Real Madrid ayaa toddobaadkan gashay xa...,bbc_somali,https://www.bbc.com/somali/articles/cd9plkkj1pdo
2,somali_news_cleaned_00003,ciyaaro,Ciyaaryahanada ugu sarreeya Afrika ee aan ka q...,Iyada oo ay soo dhowaatay isku-aadka Koobka Ad...,bbc_somali,https://www.bbc.com/somali/articles/cjdplv01z52o
3,somali_news_cleaned_00004,ciyaaro,Maxaa ugu wacan Arsenal inaysan weligeed ku gu...,Arsenal waligeed kuma guuleysan Champions Leag...,bbc_somali,https://www.bbc.com/somali/articles/c332veek778o
4,somali_news_cleaned_00005,ciyaaro,Dhamaan xogta iyo macluumaadka aad u baahan ta...,Iyadoo dhamaan kooxaha ka qayb galaya koobka a...,bbc_somali,https://www.bbc.com/somali/articles/cly07pjzdm0o


In [11]:
# CELL — SOMALI SUPERVISOR REPORT / WHATSAPP READY
# ================================================
# Run this cell AFTER the full cleaning notebook has completed.
# It prints one detailed Somali report that you can copy and send to your supervisor.

import pandas as pd
import numpy as np
from datetime import datetime

# ── Helper functions ─────────────────────────────────────────────────────────

def get_existing_df(names):
    """Return the first existing DataFrame from a list of variable names."""
    for name in names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            return globals()[name], name
    return None, None


def safe_count(df):
    return len(df) if isinstance(df, pd.DataFrame) else 0


def get_col(df, possible_cols):
    """Find the first available column name from possible options."""
    if not isinstance(df, pd.DataFrame):
        return None
    for col in possible_cols:
        if col in df.columns:
            return col
    return None


def fmt_num(n):
    """Format number with commas."""
    try:
        return f"{int(n):,}"
    except:
        return str(n)


def percent(part, total):
    if total == 0:
        return "0%"
    return f"{(part / total) * 100:.1f}%"


def count_by_column_text(df, col, title, max_items=30):
    """Create WhatsApp-friendly count lines."""
    if not isinstance(df, pd.DataFrame) or col is None or col not in df.columns:
        return f"{title}\n- Xog lama helin.\n"

    counts = (
        df[col]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
        .value_counts()
    )

    lines = [title]
    for name, count in counts.head(max_items).items():
        lines.append(f"- {name}: {fmt_num(count)}")

    if len(counts) > max_items:
        lines.append(f"- Iyo {len(counts) - max_items} kale...")

    return "\n".join(lines) + "\n"


def count_reason_text(df, col, title, empty_value=None, max_items=20):
    if not isinstance(df, pd.DataFrame) or col is None or col not in df.columns:
        return f"{title}\n- Xog lama helin.\n"

    series = df[col].fillna("").astype(str).str.strip()

    if empty_value is not None:
        series = series.replace("", empty_value)

    counts = series.value_counts()

    lines = [title]
    for name, count in counts.head(max_items).items():
        lines.append(f"- {name}: {fmt_num(count)}")

    return "\n".join(lines) + "\n"


# ── Find important DataFrames from the notebook ──────────────────────────────

raw_data, raw_name = get_existing_df([
    "raw_df",
    "combined_raw_data",
    "df_report"
])

work_data, work_name = get_existing_df([
    "work_df",
    "df_cleaned_work",
    "cleaning_work_df"
])

final_data, final_name = get_existing_df([
    "cleaned_dataset",
    "cleaned_dataset_maximum",
    "final_cleaned_dataset",
    "cleaned_export"
])

suspicious_data, suspicious_name = get_existing_df([
    "suspicious_rows_for_review",
    "suspicious_export"
])

removed_non_somali_data, removed_non_somali_name = get_existing_df([
    "removed_non_somali_articles",
    "non_somali_articles",
    "removed_english_articles"
])

removed_all_data, removed_all_name = get_existing_df([
    "removed_rows_all_reasons",
    "removed_rows",
    "removed_articles"
])


# ── Basic counts ─────────────────────────────────────────────────────────────

raw_total = safe_count(raw_data)
work_total = safe_count(work_data)
final_total = safe_count(final_data)
suspicious_total = safe_count(suspicious_data)
removed_non_somali_total = safe_count(removed_non_somali_data)
removed_all_total = safe_count(removed_all_data)

if raw_total == 0 and work_total > 0:
    raw_total = work_total

total_removed = raw_total - final_total if raw_total >= final_total else 0
kept_percent = percent(final_total, raw_total)
removed_percent = percent(total_removed, raw_total)

# Columns
raw_site_col = get_col(raw_data, ["site", "source", "source_site"])
raw_category_col = get_col(raw_data, ["category", "file_category"])

final_site_col = get_col(final_data, ["site", "source", "source_site"])
final_category_col = get_col(final_data, ["category", "file_category"])

body_col_final = get_col(final_data, ["body_clean", "body"])
headline_col_final = get_col(final_data, ["headline_clean", "headline"])


# ── Extra cleaning statistics ────────────────────────────────────────────────

non_article_removed = 0
duplicate_url_rows = 0
duplicate_body_rows = 0
short_body_rows = 0
empty_body_rows = 0
large_loss_rows = 0

if isinstance(work_data, pd.DataFrame):
    if "removal_reason" in work_data.columns:
        non_article_removed = int(work_data["removal_reason"].fillna("").astype(str).str.strip().ne("").sum())

    if "duplicate_url" in work_data.columns:
        duplicate_url_rows = int(work_data["duplicate_url"].fillna(False).sum())

    if "duplicate_clean_body" in work_data.columns:
        duplicate_body_rows = int(work_data["duplicate_clean_body"].fillna(False).sum())

    if "body_words_after" in work_data.columns:
        short_body_rows = int((pd.to_numeric(work_data["body_words_after"], errors="coerce").fillna(0) < 50).sum())

    if "body_clean" in work_data.columns:
        empty_body_rows = int(work_data["body_clean"].fillna("").astype(str).str.strip().eq("").sum())

    if "cleaning_loss_ratio" in work_data.columns:
        large_loss_rows = int((pd.to_numeric(work_data["cleaning_loss_ratio"], errors="coerce").fillna(0) > 0.65).sum())


# ── Final body statistics ────────────────────────────────────────────────────

avg_words = 0
min_words = 0
max_words = 0

if isinstance(final_data, pd.DataFrame) and body_col_final is not None:
    word_counts = final_data[body_col_final].fillna("").astype(str).str.split().str.len()

    if len(word_counts) > 0:
        avg_words = round(word_counts.mean(), 1)
        min_words = int(word_counts.min())
        max_words = int(word_counts.max())


# ── Build report text ────────────────────────────────────────────────────────

today = datetime.now().strftime("%d/%m/%Y")

report_lines = []

report_lines.append("📌 *Warbixinta Nadiifinta Dataset-ka Somali News*")
report_lines.append(f"Taariikh: {today}")
report_lines.append("")
report_lines.append("Mudane Supervisor,")
report_lines.append("")
report_lines.append(
    "Waxaan sameynay uruurin, isku-daris, falanqeyn, iyo nadiifin ballaaran "
    "oo lagu sameeyay dataset-ka wararka Af-Soomaaliga ah ee laga soo qaaday website-yo kala duwan."
)
report_lines.append("")

# Overall summary
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("1️⃣ *Xogta Guud ee Dataset-ka Ka Hor Cleaning-ka*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append(f"Waxaan soo uruurinay guud ahaan: *{fmt_num(raw_total)} maqaal / record*.")
report_lines.append(f"Website-yada/source-yada laga soo uruuriyay: *{raw_data[raw_site_col].nunique() if raw_site_col and isinstance(raw_data, pd.DataFrame) else 'N/A'} website*.")
report_lines.append(f"Qeybaha/categories-ka dataset-ka ku jiray: *{raw_data[raw_category_col].nunique() if raw_category_col and isinstance(raw_data, pd.DataFrame) else 'N/A'} category*.")
report_lines.append("")

# Source distribution before
report_lines.append(count_by_column_text(
    raw_data,
    raw_site_col,
    "📍 *Tirada maqaalada laga soo qaaday website kasta ka hor cleaning-ka:*"
))

# Category distribution before
report_lines.append(count_by_column_text(
    raw_data,
    raw_category_col,
    "📂 *Tirada maqaalada category kasta ka hor cleaning-ka:*"
))

# Cleaning explanation
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("2️⃣ *Waxyaabihii Cleaning-ka Lagu Sameeyay*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("Cleaning-ka waxaa lagu sameeyay dhowr marxaladood oo kala ah:")
report_lines.append("")
report_lines.append("✅ Isku-daridda dhammaan raw files-ka hal dataset.")
report_lines.append("✅ Standardization: columns-ka sida headline, body, category, site/source iyo URL waa la mideeyay.")
report_lines.append("✅ Ka saaridda extra spaces, line breaks, strange characters, HTML entities, iyo punctuation khaldan.")
report_lines.append("✅ Ka saaridda website boilerplate sida footer, header, social media text, advertisement text, iyo contact form text.")
report_lines.append("✅ Ka saaridda image captions sida BBC-da: ‘Xigashada Sawirka’ iyo credits la mid ah.")
report_lines.append("✅ Ka saaridda ‘Read more’, related links, source credits, author/byline text, iyo website signatures.")
report_lines.append("✅ Ka saaridda pages aan article ahayn sida Privacy Policy, Terms, Contact pages, archive pages, iyo listing pages.")
report_lines.append("✅ Ka saaridda duplicate rows iyadoo la eegayo URL iyo cleaned body.")
report_lines.append("✅ Ka saaridda articles English/non-Somali ah iyadoo la isticmaalayo Somali language checks.")
report_lines.append("✅ Kala saaridda suspicious rows si manual review loogu sameeyo.")
report_lines.append("")

# Removed stats
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("3️⃣ *Natiijada Cleaning-ka*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append(f"Raw dataset markii hore: *{fmt_num(raw_total)} records*.")
report_lines.append(f"Dataset-ka nadiifka ah ee ugu dambeeyay: *{fmt_num(final_total)} records*.")
report_lines.append(f"Records-ka la saaray guud ahaan: *{fmt_num(total_removed)} records* ({removed_percent}).")
report_lines.append(f"Records-ka la reebay/clean dataset-ka ku haray: *{fmt_num(final_total)} records* ({kept_percent}).")
report_lines.append("")

report_lines.append("📌 *Sababaha ugu waaweyn ee records loo saaray ama review loogu calaamadeeyay:*")
report_lines.append(f"- Non-article pages / boilerplate pages: {fmt_num(non_article_removed)}")
report_lines.append(f"- English ama non-Somali articles la saaray: {fmt_num(removed_non_somali_total)}")
report_lines.append(f"- Duplicate URL rows la helay: {fmt_num(duplicate_url_rows)}")
report_lines.append(f"- Duplicate cleaned body rows la helay: {fmt_num(duplicate_body_rows)}")
report_lines.append(f"- Very short body rows la helay: {fmt_num(short_body_rows)}")
report_lines.append(f"- Empty body after cleaning: {fmt_num(empty_body_rows)}")
report_lines.append(f"- Rows cleaning-ku ka jaray text aad u badan: {fmt_num(large_loss_rows)}")
report_lines.append(f"- Suspicious rows loo diray manual review: {fmt_num(suspicious_total)}")
report_lines.append("")

# Removal reasons if available
if isinstance(work_data, pd.DataFrame) and "removal_reason" in work_data.columns:
    report_lines.append(count_reason_text(
        work_data[work_data["removal_reason"].fillna("").astype(str).str.strip().ne("")],
        "removal_reason",
        "🧹 *Non-article / removal reasons faahfaahsan:*"
    ))

# Final distribution
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("4️⃣ *Dataset-ka Ugu Dambeeyay Kadib Cleaning-ka*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append(f"Cleaned dataset-ka ugu dambeeya wuxuu ka kooban yahay: *{fmt_num(final_total)} articles nadiif ah*.")
report_lines.append(f"Celceliska erayada article kasta kadib cleaning: *{avg_words} words*.")
report_lines.append(f"Article-ka ugu gaaban kadib cleaning: *{fmt_num(min_words)} words*.")
report_lines.append(f"Article-ka ugu dheer kadib cleaning: *{fmt_num(max_words)} words*.")
report_lines.append("")

report_lines.append(count_by_column_text(
    final_data,
    final_site_col,
    "📍 *Tirada maqaalada website kasta kadib cleaning-ka:*"
))

report_lines.append(count_by_column_text(
    final_data,
    final_category_col,
    "📂 *Tirada maqaalada category kasta kadib cleaning-ka:*"
))

# Site-specific patterns
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("5️⃣ *Patterns-ka Gaarka ah ee Website-yada Laga Nadiifiyay*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("Cleaning-ku wuxuu si gaar ah ula tacaalay noise-ka website kasta:")
report_lines.append("")
report_lines.append("• *BBC Somali*: image captions, ‘Xigashada Sawirka’, ‘End of content’, WhatsApp promotion, iyo advertisement markers.")
report_lines.append("• *Caasimada*: ‘Share Muqdisho (Caasimada Online)’, ‘Read more’, bylines, source credits, email text, iyo W/Q credits.")
report_lines.append("• *Goobjoog*: ‘Goobjoog News’, video/audio prompts, ‘Wariye:’, iyo footer text.")
report_lines.append("• *Mustaqbal Media*: ‘Isha:’, ‘La soco wixii kasoo kordha’, iyo trailing Mustaqbal text.")
report_lines.append("• *Laacibnet*: comment form text sida ‘Save my name, email...’ iyo footer noise.")
report_lines.append("• *Kubadlive*: view counters sida ‘Views: 9’ iyo video prompt text.")
report_lines.append("• *Puntland Post*: trailing ‘PUNTLAND POST’ iyo policy/contact pages.")
report_lines.append("• *SONNA*: SONNA datelines iyo author credits.")
report_lines.append("• *Wararka24*: source text, policy pages, iyo website boilerplate.")
report_lines.append("• *GaroweOnline*: listing pages, cookie text, copyright, iyo advertise blocks.")
report_lines.append("")

# Output files
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("6️⃣ *Files-ka La Soo Saaray*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("Natiijada cleaning-ka waxaa laga soo saaray files-kan:")
report_lines.append("")
report_lines.append("✅ cleaned_dataset_maximum.csv — dataset-ka nadiifka ah ee final-ka ah.")
report_lines.append("✅ suspicious_rows_for_review.csv — rows u baahan manual review.")
report_lines.append("✅ removed_non_somali_articles.csv — articles English/non-Somali ah oo la saaray.")
report_lines.append("✅ removed_rows_all_reasons.csv — dhammaan rows-ka la saaray iyo sababtooda.")
report_lines.append("")

# Final note
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append("7️⃣ *Gunaanad*")
report_lines.append("━━━━━━━━━━━━━━━━━━━━")
report_lines.append(
    f"Gabagabadii, dataset-ka waxaa laga soo bilaabay *{fmt_num(raw_total)} raw records*, "
    f"waxaana laga soo saaray *{fmt_num(final_total)} articles nadiif ah* oo ku habboon in loo isticmaalo "
    "training, analysis, iyo thesis-ka Automatic Somali Headline Generation."
)
report_lines.append("")
report_lines.append(
    "Cleaning-ka waxaa lagu sameeyay si taxaddar leh si aan looga saarin content muhiim ah oo Af-Soomaali ah, "
    "iyadoo la saaray kaliya noise, duplicates, non-article pages, iyo English/non-Somali articles."
)

REPORT_TEXT = "\n".join(report_lines)

print(REPORT_TEXT)

📌 *Warbixinta Nadiifinta Dataset-ka Somali News*
Taariikh: 22/05/2026

Mudane Supervisor,

Waxaan sameynay uruurin, isku-daris, falanqeyn, iyo nadiifin ballaaran oo lagu sameeyay dataset-ka wararka Af-Soomaaliga ah ee laga soo qaaday website-yo kala duwan.

━━━━━━━━━━━━━━━━━━━━
1️⃣ *Xogta Guud ee Dataset-ka Ka Hor Cleaning-ka*
━━━━━━━━━━━━━━━━━━━━
Waxaan soo uruurinay guud ahaan: *35,797 maqaal / record*.
Website-yada/source-yada laga soo uruuriyay: *12 website*.
Qeybaha/categories-ka dataset-ka ku jiray: *4 category*.

📍 *Tirada maqaalada laga soo qaaday website kasta ka hor cleaning-ka:*
- goobjoog: 12,860
- caasimada: 9,365
- kooxda: 7,193
- mustaqbalmedia: 4,640
- laacibnet: 1,094
- bbc_somali: 245
- puntlandpost: 226
- kubadlive: 114
- sonna: 26
- wararka24: 15
- garoweonline: 11
- kooxdamanta: 8

📂 *Tirada maqaalada category kasta ka hor cleaning-ka:*
- caalamka: 10,116
- siyaasad: 10,062
- ciyaaro: 9,024
- amni: 6,595

━━━━━━━━━━━━━━━━━━━━
2️⃣ *Waxyaabihii Cleaning-ka Lagu Samee

In [12]:
# SHORT FORMAL SUPERVISOR REPORT — WHATSAPP READY
# Run this after the cleaning notebook is completed.

import pandas as pd

def fmt_num(n):
    try:
        return f"{int(n):,}"
    except:
        return str(n)

def get_df(names):
    for name in names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            return globals()[name]
    return pd.DataFrame()

raw_data = get_df(["raw_df", "combined_raw_data", "df_report"])
final_data = get_df(["cleaned_dataset", "cleaned_dataset_maximum", "final_cleaned_dataset", "cleaned_export"])
removed_non_somali = get_df(["removed_non_somali_articles", "removed_english_articles", "non_somali_articles"])
suspicious_data = get_df(["suspicious_rows_for_review", "suspicious_export"])

raw_total = len(raw_data)
final_total = len(final_data)
removed_total = raw_total - final_total if raw_total >= final_total else 0

site_col = "site" if "site" in raw_data.columns else "source" if "source" in raw_data.columns else None
cat_col = "category" if "category" in raw_data.columns else None

website_count = raw_data[site_col].nunique() if site_col else "N/A"
category_count = raw_data[cat_col].nunique() if cat_col else "N/A"

message = f"""Waxaan soo arruuriney {fmt_num(raw_total)} maqaal oo raw dataset ah, kuwaas oo laga soo qaaday {website_count} website oo warar Af-Soomaali ah laga helo, waxaana dataset-ku ka koobnaa {category_count} category.

Kadib waxaan sameynay cleaning iyo preprocessing dhammeystiran, waxaana laga saaray duplicate articles, empty/short articles, website boilerplate, image captions, footer/header text, read more sections, source credits, iyo articles aan Af-Soomaali ahayn.

Natiijada ugu dambeysa, waxaa noo haray {fmt_num(final_total)} maqaal oo nadiif ah, diyaarna u ah in loo isticmaalo tababarka model-ka iyo qeybaha xiga ee thesis-ka.

Sidoo kale, {fmt_num(len(suspicious_data))} rows ayaa loo saaray manual review, halka {fmt_num(len(removed_non_somali))} articles English/non-Somali ah laga saaray dataset-ka."""

print(message)

Waxaan soo arruuriney 35,797 maqaal oo raw dataset ah, kuwaas oo laga soo qaaday 12 website oo warar Af-Soomaali ah laga helo, waxaana dataset-ku ka koobnaa 4 category.

Kadib waxaan sameynay cleaning iyo preprocessing dhammeystiran, waxaana laga saaray duplicate articles, empty/short articles, website boilerplate, image captions, footer/header text, read more sections, source credits, iyo articles aan Af-Soomaali ahayn.

Natiijada ugu dambeysa, waxaa noo haray 34,477 maqaal oo nadiif ah, diyaarna u ah in loo isticmaalo tababarka model-ka iyo qeybaha xiga ee thesis-ka.

Sidoo kale, 35,797 rows ayaa loo saaray manual review, halka 75 articles English/non-Somali ah laga saaray dataset-ka.


In [13]:
# SHORT FORMAL SUPERVISOR REPORT — SOMALI + ENGLISH
# =================================================
# Run this after the full cleaning notebook is completed.

import pandas as pd
from datetime import datetime

# ── Helpers ─────────────────────────────────────────────────────────────────

def fmt_num(n):
    try:
        return f"{int(n):,}"
    except:
        return str(n)


def get_df(names):
    """Get first existing DataFrame from notebook variables."""
    for name in names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            return globals()[name], name
    return pd.DataFrame(), None


def find_col(df, possible_cols):
    """Find a column using possible names."""
    for col in possible_cols:
        if col in df.columns:
            return col
    return None


def infer_site_from_source_file(df):
    """
    If site/source column is missing, try to infer site from source_file.
    Example: bbc_somali__ciyaaro.csv → bbc_somali
    """
    if "source_file" in df.columns and "site" not in df.columns:
        df = df.copy()
        df["site"] = (
            df["source_file"]
            .astype(str)
            .str.replace(".csv", "", regex=False)
            .str.replace(".xlsx", "", regex=False)
            .str.split("__")
            .str[0]
            .str.strip()
            .str.lower()
        )
    return df


def infer_category_from_source_file(df):
    """
    If category column is missing, try to infer category from source_file.
    Example: bbc_somali__ciyaaro.csv → ciyaaro
    """
    if "source_file" in df.columns and "category" not in df.columns:
        df = df.copy()
        df["category"] = (
            df["source_file"]
            .astype(str)
            .str.replace(".csv", "", regex=False)
            .str.replace(".xlsx", "", regex=False)
            .str.split("__")
            .str[-1]
            .str.strip()
            .str.lower()
        )
    return df


def count_lines(df, col, label_before="", max_items=50):
    """Return count lines for WhatsApp-friendly message."""
    if df.empty or col is None or col not in df.columns:
        return "- Xog lama helin."

    counts = (
        df[col]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
        .value_counts()
    )

    lines = []
    for name, count in counts.head(max_items).items():
        lines.append(f"- {name}: {fmt_num(count)}")

    return "\n".join(lines)


def before_after_category_lines(raw_df, final_df, raw_cat_col, final_cat_col):
    """Create side-by-side before/after category counts."""
    if raw_df.empty or final_df.empty or raw_cat_col is None or final_cat_col is None:
        return "- Xog lama helin."

    before_counts = (
        raw_df[raw_cat_col]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
        .value_counts()
    )

    after_counts = (
        final_df[final_cat_col]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
        .value_counts()
    )

    all_categories = sorted(set(before_counts.index).union(set(after_counts.index)))

    lines = []
    for cat in all_categories:
        before = before_counts.get(cat, 0)
        after = after_counts.get(cat, 0)
        removed = before - after
        lines.append(
            f"- {cat}: before {fmt_num(before)} → after {fmt_num(after)} "
            f"(removed {fmt_num(removed)})"
        )

    return "\n".join(lines)


# ── Get DataFrames from notebook ─────────────────────────────────────────────

raw_data, raw_name = get_df([
    "raw_df",
    "combined_raw_data",
    "df_report"
])

final_data, final_name = get_df([
    "cleaned_dataset_maximum",
    "cleaned_dataset",
    "final_cleaned_dataset",
    "cleaned_export"
])

removed_non_somali, _ = get_df([
    "removed_non_somali_articles",
    "removed_english_articles",
    "non_somali_articles"
])

suspicious_data, _ = get_df([
    "suspicious_rows_for_review",
    "suspicious_export"
])

raw_data = infer_site_from_source_file(raw_data)
raw_data = infer_category_from_source_file(raw_data)

final_data = infer_site_from_source_file(final_data)
final_data = infer_category_from_source_file(final_data)

raw_site_col = find_col(raw_data, ["site", "source", "source_site"])
raw_cat_col = find_col(raw_data, ["category", "file_category"])

final_site_col = find_col(final_data, ["site", "source", "source_site"])
final_cat_col = find_col(final_data, ["category", "file_category"])


# ── Main numbers ─────────────────────────────────────────────────────────────

raw_total = len(raw_data)
final_total = len(final_data)
removed_total = raw_total - final_total if raw_total >= final_total else 0

website_count = raw_data[raw_site_col].nunique() if raw_site_col else "N/A"
category_count = raw_data[raw_cat_col].nunique() if raw_cat_col else "N/A"

non_somali_total = len(removed_non_somali)
suspicious_total = len(suspicious_data)

today = datetime.now().strftime("%d/%m/%Y")


# ── Somali Report ────────────────────────────────────────────────────────────

somali_message = f"""Waxaan soo arruuriney {fmt_num(raw_total)} maqaal oo raw dataset ah, kuwaas oo laga soo qaaday {website_count} website oo warar Af-Soomaali ah laga helo. Dataset-ku wuxuu ka koobnaa {category_count} category.

Ka hor cleaning-ka, tirada maqaalada website kasta laga soo qaaday waxay ahayd:
{count_lines(raw_data, raw_site_col)}

Category-yada dataset-ka ka hor iyo kadib cleaning-ka:
{before_after_category_lines(raw_data, final_data, raw_cat_col, final_cat_col)}

Kadib cleaning iyo preprocessing, waxaan ka saarnay duplicate articles, empty/short articles, website boilerplate, image captions, read more sections, footer/header text, source credits, iyo articles English/non-Somali ah.

Natiijada ugu dambeysa, waxaa noo haray {fmt_num(final_total)} maqaal oo nadiif ah, diyaarna u ah training-ka model-ka iyo qeybaha xiga ee thesis-ka.

Guud ahaan waxaa laga saaray {fmt_num(removed_total)} records. Waxaa ka mid ah {fmt_num(non_somali_total)} articles English/non-Somali ah, halka {fmt_num(suspicious_total)} rows loo saaray manual review."""


# ── English Report ───────────────────────────────────────────────────────────

english_message = f"""We collected {fmt_num(raw_total)} raw news articles from {website_count} Somali news websites. The dataset contained {category_count} categories.

Before cleaning, the number of articles collected from each website was:
{count_lines(raw_data, raw_site_col)}

Category distribution before and after cleaning:
{before_after_category_lines(raw_data, final_data, raw_cat_col, final_cat_col)}

After cleaning and preprocessing, we removed duplicate articles, empty/short articles, website boilerplate, image captions, read more sections, footer/header text, source credits, and English/non-Somali articles.

The final cleaned dataset contains {fmt_num(final_total)} clean articles, ready for model training and the next stages of the thesis.

In total, {fmt_num(removed_total)} records were removed. This includes {fmt_num(non_somali_total)} English/non-Somali articles, while {fmt_num(suspicious_total)} rows were separated for manual review."""


# ── Print both versions ──────────────────────────────────────────────────────

print("🇸🇴 SOMALI VERSION")
print("═" * 70)
print(somali_message)

print("\n\n🇬🇧 ENGLISH VERSION")
print("═" * 70)
print(english_message)

🇸🇴 SOMALI VERSION
══════════════════════════════════════════════════════════════════════
Waxaan soo arruuriney 35,797 maqaal oo raw dataset ah, kuwaas oo laga soo qaaday 12 website oo warar Af-Soomaali ah laga helo. Dataset-ku wuxuu ka koobnaa 4 category.

Ka hor cleaning-ka, tirada maqaalada website kasta laga soo qaaday waxay ahayd:
- goobjoog: 12,860
- caasimada: 9,365
- kooxda: 7,193
- mustaqbalmedia: 4,640
- laacibnet: 1,094
- bbc_somali: 245
- puntlandpost: 226
- kubadlive: 114
- sonna: 26
- wararka24: 15
- garoweonline: 11
- kooxdamanta: 8

Category-yada dataset-ka ka hor iyo kadib cleaning-ka:
- amni: before 6,595 → after 6,270 (removed 325)
- caalamka: before 10,116 → after 10,008 (removed 108)
- ciyaaro: before 9,024 → after 8,898 (removed 126)
- siyaasad: before 10,062 → after 9,301 (removed 761)

Kadib cleaning iyo preprocessing, waxaan ka saarnay duplicate articles, empty/short articles, website boilerplate, image captions, read more sections, footer/header text, source cr